# Lesson 17: SQL basics: SELECT, WHERE, GROUP BY · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 17

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Big Books orders

Use pd.read_sql to select order_id and unit_price_usd for orders in the Books category with exactly 4 units. Store the result in big_books.

<details><summary>Hint</summary>

Text values go in single quotes inside the query: WHERE category = 'Books' AND units = 4.

</details>

In [ ]:
import sqlite3
import pandas as pd

orders = pd.read_csv(DATA_URL.replace("loans.csv", "global/orders.csv"))
customers = pd.read_csv(DATA_URL.replace("loans.csv", "global/customers.csv"))
orders["revenue"] = (orders["units"] * orders["unit_price_usd"] * (1 - orders["discount"])).round(2)
con = sqlite3.connect(":memory:")
orders.to_sql("orders", con, index=False)
customers.to_sql("customers", con, index=False)

big_books = ...   # pd.read_sql("SELECT ... FROM orders WHERE ...", con)

print(big_books)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "assert isinstance(big_books, pd.DataFrame), \"big_books should be the DataFrame returned by pd.read_sql(query, con)\"\nassert \"order_id\" in big_books.columns and \"unit_price_usd\" in big_books.columns, \"SELECT order_id and unit_price_usd\"\n_want = orders[(orders[\"category\"] == \"Books\") & (orders[\"units\"] == 4)]\nassert sorted(big_books[\"order_id\"]) == sorted(_want[\"order_id\"]), \"Keep only rows WHERE category = 'Books' AND units = 4\"\n")

## Exercise 2: Countries with the most customers

Query the customers table: count customers per country, sort largest first, and keep the top 3. Name the count column n_customers and store the result in top_countries.

<details><summary>Hint</summary>

GROUP BY country makes one row per country; ORDER BY n_customers DESC then LIMIT 3.

</details>

In [ ]:
import sqlite3
import pandas as pd

orders = pd.read_csv(DATA_URL.replace("loans.csv", "global/orders.csv"))
customers = pd.read_csv(DATA_URL.replace("loans.csv", "global/customers.csv"))
orders["revenue"] = (orders["units"] * orders["unit_price_usd"] * (1 - orders["discount"])).round(2)
con = sqlite3.connect(":memory:")
orders.to_sql("orders", con, index=False)
customers.to_sql("customers", con, index=False)

top_countries = ...   # SELECT country, COUNT(*) AS n_customers FROM customers GROUP BY ...

print(top_countries)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "assert isinstance(top_countries, pd.DataFrame), \"top_countries should be the DataFrame returned by pd.read_sql(query, con)\"\nassert \"country\" in top_countries.columns and \"n_customers\" in top_countries.columns, \"SELECT country and COUNT(*) AS n_customers\"\nassert len(top_countries) == 3, \"Use LIMIT 3 to keep three rows\"\n_want = customers[\"country\"].value_counts().head(3)\nassert list(top_countries[\"country\"]) == list(_want.index), \"Sort with ORDER BY n_customers DESC so the biggest countries come first\"\nassert [int(v) for v in top_countries[\"n_customers\"]] == [int(v) for v in _want.values], \"n_customers should be COUNT(*) for each country\"\n")

## Exercise 3: Revenue by segment with a JOIN

Join orders to customers on customer_id and total the revenue for each customer segment. Store the result in by_segment, with the columns segment and revenue.

<details><summary>Hint</summary>

FROM orders AS o JOIN customers AS c ON o.customer_id = c.customer_id, then GROUP BY c.segment.

</details>

In [ ]:
import sqlite3
import pandas as pd

orders = pd.read_csv(DATA_URL.replace("loans.csv", "global/orders.csv"))
customers = pd.read_csv(DATA_URL.replace("loans.csv", "global/customers.csv"))
orders["revenue"] = (orders["units"] * orders["unit_price_usd"] * (1 - orders["discount"])).round(2)
con = sqlite3.connect(":memory:")
orders.to_sql("orders", con, index=False)
customers.to_sql("customers", con, index=False)

by_segment = ...   # SELECT c.segment, SUM(o.revenue) AS revenue FROM orders AS o JOIN ...

print(by_segment)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "assert isinstance(by_segment, pd.DataFrame), \"by_segment should be the DataFrame returned by pd.read_sql(query, con)\"\nassert \"segment\" in by_segment.columns and \"revenue\" in by_segment.columns, \"SELECT c.segment and SUM(o.revenue) AS revenue\"\n_want = orders.merge(customers, on=\"customer_id\").groupby(\"segment\")[\"revenue\"].sum()\nassert len(by_segment) == len(_want), \"GROUP BY c.segment should give one row per segment\"\n_got = by_segment.set_index(\"segment\")[\"revenue\"]\nfor _k in _want.index:\n    assert abs(float(_got[_k]) - _want[_k]) < 1, f\"revenue for {_k} should be the total revenue of its customers' orders\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=17) and take the quiz.